# Train all models — standalone

Self-contained: every utility, dataset loader, and model (GraphSAGE, Improved
GraphSAGE, SA-GAT, GAMLP, Retrieval-Guided GAMLP) is defined directly in the
cells below. This notebook does **not** import anything from `common/`,
`graphsage/`, `sagat/`, or `gamlp/` — copy just this one file (plus
`split_idx.csv` and, optionally, a cached `amazon_electronics_computers.npz` under
`data/`) and it runs on its own, e.g. on Kaggle.

Two simplifications versus the standalone scripts, to keep this one notebook a
reasonable size:
- GraphSAGE / Improved GraphSAGE run **full-batch only** here (no mini-batch
  neighbour sampling / importance sampling) — see `graphsage/train_graphsage.py
  --sampling` for that mode.
- Everything else (multi-aggregator, JK, SA-GAT's structural attention +
  correlation analysis, GAMLP plain/RLU, retrieval-guided GAMLP) is the full
  implementation, not a trimmed-down demo.

Set `SMOKE_TEST = False` below for full training; it defaults to `True` (a
couple epochs per model) so the whole notebook can run top-to-bottom quickly
to confirm everything works before committing to a long run.

In [ ]:
# Fresh environment (Kaggle / Colab / a clean venv) that doesn't have these yet?
# Uncomment and run once — everything below only needs these, nothing else from this repo.
# %pip install -q torch networkx scipy pandas matplotlib scikit-learn
# %pip install -q faiss-cpu  # optional: retrieval-guided GAMLP falls back to an exact torch search without it
# %pip install -q torch-geometric  # optional: only used to auto-download the dataset if data/ is empty

In [ ]:
import gc
import json
import logging
import sys
import time
from datetime import datetime
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import pearsonr
from sklearn.linear_model import LogisticRegression

try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_ROOT = REPO_ROOT / 'data'
OUTPUTS_ROOT = REPO_ROOT / 'outputs'
SPLIT_FILE = REPO_ROOT / 'split_idx.csv'

SMOKE_TEST = True  # False => full training runs (can take a long time, especially on CPU)


def epochs_for(full_epochs, smoke_epochs=3):
    return smoke_epochs if SMOKE_TEST else full_epochs


logging.basicConfig(level=logging.INFO, format='%(message)s')

## 1. Shared utilities

Generic helpers used by every model below: seeding, output-dir/logging
conventions, and plain-PyTorch graph ops (`scatter_add` / `scatter_softmax` /
...) that stand in for `torch-scatter` / `torch-sparse` / `torch-geometric.nn`
so nothing here needs a compiled extension.

In [ ]:
def set_seed(seed):
    """Seed Python/NumPy/PyTorch RNGs and force deterministic cuDNN, for reproducible runs."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_device(gpu):
    """Pick a CUDA device if available and requested, else CPU."""
    if gpu >= 0 and torch.cuda.is_available():
        return torch.device(f'cuda:{gpu}')
    return torch.device('cpu')


def make_output_dir(base_dir, run_name):
    """Create (and return) outputs/<family>/<run_name>/ for one training run."""
    out_dir = Path(base_dir) / run_name
    out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir


def setup_logger(name, out_dir):
    """Build a logger that writes to both stdout and <out_dir>/train.log."""
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    fmt = logging.Formatter('%(asctime)s | %(levelname)s | %(message)s')
    for h in [logging.StreamHandler(sys.stdout), logging.FileHandler(out_dir / 'train.log')]:
        h.setFormatter(fmt)
        logger.addHandler(h)
    return logger


def write_json(path, payload):
    """Write a dict to disk as pretty-printed JSON (used for results.json)."""
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(payload, f, indent=2, sort_keys=True)


def append_jsonl(path, payload):
    """Append one JSON record as a line to a .jsonl file (used for per-epoch metrics)."""
    with open(path, 'a', encoding='utf-8') as f:
        f.write(json.dumps(payload, sort_keys=True) + '\n')


def count_params(model):
    """Total number of trainable parameters in a model."""
    return sum(p.numel() for p in model.parameters())


def plot_training_curves(metrics_path, out_dir, title=None):
    """Plot loss + train/val accuracy from metrics.jsonl and save/show curves.png."""
    metrics_path = Path(metrics_path)
    if not metrics_path.exists():
        return None
    records = [json.loads(line) for line in metrics_path.read_text(encoding='utf-8').splitlines() if line.strip()]
    if not records:
        return None

    xs = list(range(len(records)))
    loss = [r.get('loss') for r in records]
    train_acc = [r.get('train_acc') for r in records]
    val_pts = [(i, r['val_acc']) for i, r in enumerate(records) if r.get('val_acc') is not None]
    best_test = records[-1].get('best_test')
    stage_starts = [i for i in range(1, len(records)) if records[i].get('stage') != records[i - 1].get('stage')]

    fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(11, 4))
    ax_loss.plot(xs, loss, color='tab:red', linewidth=1.2)
    ax_loss.set_xlabel('Epoch'); ax_loss.set_ylabel('Train loss'); ax_loss.set_title('Loss')
    ax_acc.plot(xs, train_acc, color='tab:blue', linewidth=1.2, label='train')
    if val_pts:
        vx, vy = zip(*val_pts)
        ax_acc.plot(vx, vy, color='tab:orange', linewidth=1.2, label='valid')
        bi = max(range(len(vy)), key=lambda k: vy[k])
        ax_acc.scatter([vx[bi]], [vy[bi]], color='tab:orange', zorder=3, label=f'best valid = {vy[bi]:.4f}')
    if best_test is not None and best_test >= 0:
        ax_acc.axhline(best_test, color='tab:green', linestyle='--', linewidth=1, label=f'best test = {best_test:.4f}')
    ax_acc.set_xlabel('Epoch'); ax_acc.set_ylabel('Accuracy'); ax_acc.set_title('Accuracy')
    ax_acc.legend(loc='lower right', fontsize=8)
    for ax in (ax_loss, ax_acc):
        ax.grid(alpha=0.3)
        for s in stage_starts:
            ax.axvline(s, color='gray', linestyle=':', linewidth=1)
    if title:
        fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(Path(out_dir) / 'curves.png', dpi=150)
    plt.show()
    plt.close(fig)
    return Path(out_dir) / 'curves.png'

In [ ]:
def scatter_add(src, index, dim_size):
    """Sum rows of src into `dim_size` buckets given by `index` -- the core primitive every graph op here is built from."""
    shape = (dim_size,) + src.shape[1:]
    out = src.new_zeros(shape)
    idx = index.view(-1, *([1] * (src.dim() - 1))).expand_as(src)
    return out.scatter_add_(0, idx, src)


def scatter_mean(src, index, dim_size):
    """Per-bucket mean of src rows (mean-aggregation over a node's neighbours)."""
    summed = scatter_add(src, index, dim_size)
    count = scatter_add(src.new_ones(src.size(0), 1), index, dim_size).clamp(min=1)
    return summed / count


def scatter_softmax(src, index, dim_size):
    """Softmax of src within each group defined by index (edge-softmax for attention)."""
    src_max = src.new_full((dim_size,), float('-inf'))
    src_max = src_max.scatter_reduce(0, index, src, reduce='amax', include_self=True)
    src_max = torch.nan_to_num(src_max, neginf=0.0)
    shifted = (src - src_max[index]).exp()
    denom = scatter_add(shifted.unsqueeze(-1), index, dim_size).squeeze(-1).clamp(min=1e-16)
    return shifted / denom[index]


def add_self_loops(edge_index, num_nodes):
    """Append one self-loop edge (i, i) per node to edge_index."""
    loop = torch.arange(num_nodes, device=edge_index.device)
    return torch.cat([edge_index, torch.stack([loop, loop], dim=0)], dim=1)


def scatter_max_feat(src, index, dim_size):
    """Per-bucket element-wise max of src rows (max-aggregation over neighbours)."""
    out = src.new_zeros(dim_size, src.size(-1))
    out.scatter_reduce_(0, index.unsqueeze(-1).expand_as(src), src, reduce='amax', include_self=False)
    return out


def scatter_std_feat(src, index, dim_size, mean=None):
    """Per-bucket standard deviation of src rows (dispersion-aggregation over neighbours)."""
    if mean is None:
        mean = scatter_mean(src, index, dim_size)
    sq_mean = scatter_mean(src * src, index, dim_size)
    # eps keeps variance > 0: sqrt's gradient at 0 is infinite -> NaN loss otherwise
    return ((sq_mean - mean * mean).clamp(min=0) + 1e-6).sqrt()


def propagate_mean(edge_index, num_nodes, feat, deg=None, hops=1):
    """Repeated 1-hop mean propagation over the graph, `hops` times (used to precompute GAMLP's K-hop features)."""
    # edge_index is always symmetric here (see load_products), so aggregating
    # "row"-into or "col"-into buckets gives the same result either way.
    row, col = edge_index
    if deg is None:
        deg = scatter_add(row.new_ones(row.size(0), 1, dtype=torch.float32), row, num_nodes).clamp(min=1)
    for _ in range(hops):
        feat = scatter_add(feat[col], row, num_nodes) / deg
    return feat


def gcn_norm_edge_weight(edge_index, num_nodes):
    """Symmetric D^-1/2 A D^-1/2 edge weights (Kipf & Welling GCN normalisation)."""
    row, col = edge_index
    deg = scatter_add(row.new_ones(row.size(0), 1, dtype=torch.float32), row, num_nodes).squeeze(-1)
    deg_inv_sqrt = deg.pow(-0.5)
    deg_inv_sqrt[torch.isinf(deg_inv_sqrt)] = 0.0
    return deg_inv_sqrt[row] * deg_inv_sqrt[col]


def weighted_propagate(x, edge_index, edge_weight, num_nodes):
    """(A_hat @ x) using explicit per-edge weights instead of a uniform neighbour average."""
    row, col = edge_index
    return scatter_add(x[col] * edge_weight.unsqueeze(-1), row, num_nodes)

## 2. Dataset — Amazon Computers

Loaded once and reused by every model below. `split_idx.csv` at the repo root
is a fixed stratified 60/20/20 split shared by every training script in this
repo, so results here are directly comparable to CLI runs.

In [ ]:
def build_stratified_split(labels, train_ratio=0.6, valid_ratio=0.2, seed=42):
    """Stratified 60/20/20 train/valid/test split, sampled independently per class."""
    rng = np.random.default_rng(seed)
    labels_np = labels.cpu().numpy()
    split_parts = {'train': [], 'valid': [], 'test': []}
    for cls in np.unique(labels_np):
        cls_idx = np.where(labels_np == cls)[0]
        rng.shuffle(cls_idx)
        n_total = len(cls_idx)
        n_train = int(n_total * train_ratio)
        n_valid = int(n_total * valid_ratio)
        if n_train + n_valid >= n_total and n_total >= 3:
            n_train = max(1, n_total - 2)
            n_valid = 1
        elif n_train + n_valid >= n_total and n_total == 2:
            n_train = 1
            n_valid = 1
        elif n_total == 1:
            n_train = 1
            n_valid = 0
        split_parts['train'].append(cls_idx[:n_train])
        split_parts['valid'].append(cls_idx[n_train:n_train + n_valid])
        split_parts['test'].append(cls_idx[n_train + n_valid:])
    split_idx = {}
    for name, parts in split_parts.items():
        idx = np.concatenate([p for p in parts if len(p) > 0]) if any(len(p) > 0 for p in parts) else np.array([], dtype=np.int64)
        rng.shuffle(idx)
        split_idx[name] = torch.from_numpy(idx).long()
    return split_idx


def find_amazon_computer_npz(root):
    """Locate (or auto-download) the Amazon Computers .npz file under `root`."""
    root = Path(root)
    candidates = sorted(root.glob('amazon_co_buy_computer*/amazon_electronics_computers.npz'))
    candidates += sorted(root.glob('**/amazon_electronics_computers.npz'))
    if not candidates:
        try:
            from torch_geometric.datasets import Amazon
            Amazon(root=str(root), name='Computers')
        except Exception as e:
            raise FileNotFoundError(
                f'amazon_electronics_computers.npz not found under {root} and auto-download failed: {e}\n'
                'Place the file manually, or `pip install torch-geometric` to auto-download.'
            ) from e
        candidates = sorted(root.glob('amazon_co_buy_computer*/amazon_electronics_computers.npz'))
        candidates += sorted(root.glob('**/amazon_electronics_computers.npz'))
    if not candidates:
        raise FileNotFoundError(f'amazon_electronics_computers.npz not found under {root}')
    return candidates[0]


def load_products(root, logger, split_seed=42):
    """Load the Amazon Computers graph, features, labels and a default split from the .npz file."""
    npz_path = find_amazon_computer_npz(root)
    with np.load(npz_path, allow_pickle=True) as loader:
        loader = dict(loader)
    adj = sp.csr_matrix((loader['adj_data'], loader['adj_indices'], loader['adj_indptr']), shape=tuple(loader['adj_shape']))
    adj = adj.maximum(adj.T).tocoo()
    edge_index = torch.from_numpy(np.vstack([adj.row, adj.col]).astype(np.int64, copy=False))
    if 'attr_data' in loader:
        x = sp.csr_matrix((loader['attr_data'], loader['attr_indices'], loader['attr_indptr']), shape=tuple(loader['attr_shape'])).toarray()
    else:
        x = loader['attr_matrix']
    labels = torch.as_tensor(loader['labels'], dtype=torch.long).view(-1)
    num_classes = int(len(np.unique(loader['labels'])))
    split_idx = build_stratified_split(labels, seed=split_seed)
    data = SimpleNamespace(x=torch.as_tensor(x, dtype=torch.float32), edge_index=edge_index.long(), num_nodes=int(labels.numel()))
    logger.info(
        'Loaded Amazon Computers from %s | nodes=%d edges=%d train=%d valid=%d test=%d classes=%d',
        npz_path, data.num_nodes, data.edge_index.size(1),
        split_idx['train'].numel(), split_idx['valid'].numel(), split_idx['test'].numel(), num_classes,
    )
    return data, labels, split_idx, num_classes


class AccuracyEvaluator:
    """Minimal accuracy-only evaluator matching the {y_true, y_pred} -> {'acc': ...} OGB-style interface."""
    def eval(self, input_dict):
        y_true = input_dict['y_true'].view(-1)
        y_pred = input_dict['y_pred'].view(-1)
        return {'acc': float((y_true == y_pred).float().mean().item())}


def load_split_idx_csv(path):
    """Load a fixed split from split_idx.csv, so every model evaluates on the same nodes."""
    parts = {'train': [], 'valid': [], 'test': []}
    with open(path, newline='', encoding='utf-8') as f:
        next(f)
        for line in f:
            line = line.strip()
            if not line:
                continue
            split, idx = line.split(',', 1)
            parts[split].append(int(idx))
    return {k: torch.tensor(v, dtype=torch.long) for k, v in parts.items()}

In [ ]:
logger = logging.getLogger('amazon-gnn-benchmark')

DEVICE = get_device(0)
print('device:', DEVICE)

data, labels, split_idx, num_classes = load_products(DATA_ROOT, logger, split_seed=0)
if SPLIT_FILE.is_file():
    split_idx = load_split_idx_csv(SPLIT_FILE)
    logger.info('Using fixed split from %s', SPLIT_FILE)

evaluator = AccuracyEvaluator()
IN_DIM = data.x.size(1)
results = []  # collects one dict per run_x(...) call below, for the summary table at the end

## 3. GraphSAGE (mean aggregator, full-batch)

In [ ]:
class SAGELayer(nn.Module):
    """One GraphSAGE layer: self-transform + mean-neighbour-transform, summed."""
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)

    def forward(self, x, edge_index, num_nodes):
        row, col = edge_index
        neigh_mean = scatter_mean(x[col], row, num_nodes)
        return self.lin_self(x) + self.lin_neigh(neigh_mean)


class GraphSAGE(nn.Module):
    """Stack of SAGELayers with BatchNorm + ReLU + dropout between hidden layers."""
    def __init__(self, in_dim, hidden, out_dim, num_layers, dropout):
        super().__init__()
        dims = [in_dim] + [hidden] * (num_layers - 1) + [out_dim]
        self.convs = nn.ModuleList([SAGELayer(dims[i], dims[i + 1]) for i in range(num_layers)])
        self.bns = nn.ModuleList([nn.BatchNorm1d(hidden) for _ in range(num_layers - 1)])
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index, num_nodes):
        for i, conv in enumerate(self.convs[:-1]):
            x = conv(x, edge_index, num_nodes)
            x = self.dropout(F.relu(self.bns[i](x)))
        return self.convs[-1](x, edge_index, num_nodes)

In [ ]:
def train_epoch_sage(model, x, edge_index, num_nodes, labels, train_idx, optimizer, evaluator):
    """One full-batch training step (forward, loss, backward) + train accuracy."""
    model.train()
    optimizer.zero_grad()
    out = model(x, edge_index, num_nodes)
    loss = F.cross_entropy(out[train_idx], labels[train_idx])
    loss.backward()
    optimizer.step()
    pred = out[train_idx].argmax(dim=-1).detach().cpu()
    acc = evaluator.eval({'y_true': labels[train_idx].cpu().view(-1, 1), 'y_pred': pred.view(-1, 1)})['acc']
    return float(loss.item()), acc


@torch.no_grad()
def evaluate_sage(model, x, edge_index, num_nodes, labels, idx, evaluator):
    """Full-batch forward pass + accuracy on a given node index set."""
    model.eval()
    out = model(x, edge_index, num_nodes)
    pred = out[idx].argmax(dim=-1).cpu()
    return evaluator.eval({'y_true': labels[idx].cpu().view(-1, 1), 'y_pred': pred.view(-1, 1)})['acc']


def run_graphsage(args, run_id, device, model_cls=GraphSAGE, family='graphsage'):
    """Full training loop (with early stopping) for GraphSAGE or ImprovedGraphSAGE -- shared because both models take the same (x, edge_index, num_nodes) forward signature."""
    # Shared by GraphSAGE and Improved GraphSAGE: both models take (x, edge_index,
    # num_nodes) and only differ in construction, so one training loop covers both.
    run_name = f"{datetime.now().strftime('%Y%m%d_%H%M%S')}_run{run_id}_{family}"
    out_dir = make_output_dir(OUTPUTS_ROOT / family, run_name)
    run_logger = setup_logger(family, out_dir)
    run_logger.info('Args: %s', json.dumps(vars(args), sort_keys=True, default=str))
    set_seed(args.seed + run_id)

    x = data.x.float().to(device)
    edge_index = data.edge_index.to(device)
    labels_dev = labels.to(device)
    split_dev = {k: v.to(device) for k, v in split_idx.items()}

    model = args.build_model(x.size(1), num_classes).to(device)
    run_logger.info('Model params: %d', count_params(model))
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)

    best_val = best_test = -1.0
    best_epoch = stale = 0
    metrics_path = out_dir / 'metrics.jsonl'

    for epoch in range(args.epochs):
        t0 = time.time()
        loss, train_acc = train_epoch_sage(model, x, edge_index, data.num_nodes, labels_dev, split_dev['train'], optimizer, evaluator)
        val_acc = evaluate_sage(model, x, edge_index, data.num_nodes, labels_dev, split_dev['valid'], evaluator)
        if val_acc > best_val:
            best_val, best_epoch = val_acc, epoch
            best_test = evaluate_sage(model, x, edge_index, data.num_nodes, labels_dev, split_dev['test'], evaluator)
            torch.save({'model_state': model.state_dict()}, out_dir / 'best.pt')
            stale = 0
        else:
            stale += 1
        append_jsonl(metrics_path, {
            'epoch': epoch, 'loss': loss, 'train_acc': train_acc, 'val_acc': val_acc,
            'best_val': best_val, 'best_test': best_test, 'best_epoch': best_epoch, 'time_sec': time.time() - t0,
        })
        if stale >= args.patience:
            run_logger.info('Early stopping at epoch=%d', epoch)
            break

    result = {'run': run_id, 'model_family': family, 'num_params': count_params(model), 'best_val': best_val, 'best_test': best_test, 'best_epoch': best_epoch, 'output_dir': str(out_dir)}
    write_json(out_dir / 'results.json', result)
    plot_training_curves(metrics_path, out_dir, title=family)
    run_logger.info('Final: %s', result)
    print(f"{family}: best_val={best_val:.4f}  best_test={best_test:.4f}")
    return result

In [ ]:
args_sage = SimpleNamespace(
    hidden=64, num_layers=2, dropout=0.5,
    epochs=epochs_for(200), lr=0.01, weight_decay=5e-4, patience=30,
    seed=0,
    build_model=lambda in_dim, out_dim: GraphSAGE(in_dim, 64, out_dim, 2, 0.5),
)
res_sage = run_graphsage(args_sage, run_id=0, device=DEVICE, model_cls=GraphSAGE, family='graphsage')
results.append(res_sage)

## 4. Improved GraphSAGE (multi-aggregator + Jumping Knowledge)

Reuses `train_epoch_sage` / `evaluate_sage` / `run_graphsage` above — the
model's `forward(x, edge_index, num_nodes)` signature matches `GraphSAGE`
exactly, only the architecture (and `args.build_model`) differs.

In [ ]:
class MultiAggSAGELayer(nn.Module):
    """GraphSAGE layer with mean+max+std multi-aggregation (PNA-style) instead of mean only."""
    def __init__(self, in_dim, out_dim, use_multi_aggr=True, dropout=0.0):
        super().__init__()
        self.use_multi_aggr = use_multi_aggr
        n_aggr = 3 if use_multi_aggr else 1
        self.lin = nn.Linear(in_dim * (1 + n_aggr), out_dim)
        self.bn = nn.BatchNorm1d(out_dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, edge_index, num_nodes):
        row, col = edge_index
        neigh_mean = scatter_mean(x[col], row, num_nodes)
        if self.use_multi_aggr:
            neigh_max = scatter_max_feat(x[col], row, num_nodes)
            neigh_std = scatter_std_feat(x[col], row, num_nodes, mean=neigh_mean)
            h = torch.cat([x, neigh_mean, neigh_max, neigh_std], dim=-1)
        else:
            h = torch.cat([x, neigh_mean], dim=-1)
        return self.drop(F.relu(self.bn(self.lin(h))))


class ImprovedGraphSAGE(nn.Module):
    """Stack of MultiAggSAGELayers with Jumping Knowledge combining every layer's output."""
    def __init__(self, in_dim, hidden, out_dim, num_layers, dropout, use_multi_aggr=True, jk_mode='concat'):
        super().__init__()
        self.jk_mode = jk_mode
        self.convs = nn.ModuleList([
            MultiAggSAGELayer(in_dim if i == 0 else hidden, hidden, use_multi_aggr=use_multi_aggr, dropout=dropout)
            for i in range(num_layers)
        ])
        if jk_mode == 'concat':
            self.jk_proj = nn.Linear(num_layers * hidden, hidden)
            self.jk_bn = nn.BatchNorm1d(hidden)
        self.classifier = nn.Linear(hidden, out_dim)

    def forward(self, x, edge_index, num_nodes):
        layer_outs = []
        for conv in self.convs:
            x = conv(x, edge_index, num_nodes)
            layer_outs.append(x)
        if self.jk_mode == 'concat':
            x = F.relu(self.jk_bn(self.jk_proj(torch.cat(layer_outs, dim=-1))))
        elif self.jk_mode == 'maxpool':
            x = torch.stack(layer_outs, dim=0).max(dim=0).values
        else:
            x = layer_outs[-1]
        return self.classifier(x)

In [ ]:
args_isage = SimpleNamespace(
    hidden=128, num_layers=3, dropout=0.4,
    epochs=epochs_for(300), lr=0.005, weight_decay=5e-4, patience=40,
    seed=0,
    build_model=lambda in_dim, out_dim: ImprovedGraphSAGE(in_dim, 128, out_dim, 3, 0.4, use_multi_aggr=True, jk_mode='concat'),
)
res_isage = run_graphsage(args_isage, run_id=0, device=DEVICE, model_cls=ImprovedGraphSAGE, family='improved_graphsage')
results.append(res_isage)

## 5. SA-GAT — vanilla GAT baseline vs. structure-aware attention

Degree / PageRank / clustering coefficient / betweenness centrality are
injected as an additive bias inside the attention logit (`sagat` variant),
compared against a plain GAT baseline (`gat` variant) with no structural
information at all.

In [ ]:
def log_zscore(x):
    """log1p then z-score -- normalisation for heavy-tailed structural stats (degree, betweenness)."""
    x = np.log1p(x)
    return (x - x.mean()) / (x.std() + 1e-8)


def zscore(x):
    """Plain z-score normalisation for bounded structural stats (PageRank, clustering)."""
    return (x - x.mean()) / (x.std() + 1e-8)


def compute_structural_features(edge_index, num_nodes, betweenness_samples, seed, logger):
    """Compute per-node degree/PageRank/clustering/betweenness via networkx, normalised."""
    row, col = edge_index.cpu().numpy()
    g = nx.Graph()
    g.add_nodes_from(range(num_nodes))
    g.add_edges_from(zip(row.tolist(), col.tolist()))
    g.remove_edges_from(nx.selfloop_edges(g))

    t0 = time.time()
    degree_dict = dict(g.degree())
    degree = np.array([degree_dict[i] for i in range(num_nodes)], dtype=np.float32)
    pagerank_dict = nx.pagerank(g, alpha=0.85, max_iter=100, tol=1e-6)
    pagerank = np.array([pagerank_dict[i] for i in range(num_nodes)], dtype=np.float32)
    clustering_dict = nx.clustering(g)
    clustering = np.array([clustering_dict[i] for i in range(num_nodes)], dtype=np.float32)
    betweenness_dict = nx.betweenness_centrality(g, k=min(betweenness_samples, num_nodes), seed=seed, normalized=True)
    betweenness = np.array([betweenness_dict[i] for i in range(num_nodes)], dtype=np.float32)
    logger.info('Structural features computed in %.1fs', time.time() - t0)

    feats = np.stack([log_zscore(degree), zscore(pagerank), zscore(clustering), log_zscore(betweenness)], axis=1)
    raw = {'degree': degree, 'pagerank': pagerank, 'clustering': clustering, 'betweenness': betweenness}
    return torch.from_numpy(feats).float(), raw

In [ ]:
class GATLayer(nn.Module):
    """One GAT attention layer, with an optional additive structural-attention bias (SA-GAT)."""
    def __init__(self, in_dim, head_dim, heads, dropout, att_dropout, negative_slope, concat=True, struct_dim=None):
        super().__init__()
        self.heads = heads
        self.head_dim = head_dim
        self.concat = concat
        self.struct_dim = struct_dim
        self.lin = nn.Linear(in_dim, heads * head_dim, bias=False)
        self.att_src = nn.Parameter(torch.empty(heads, head_dim))
        self.att_dst = nn.Parameter(torch.empty(heads, head_dim))
        self.bias = nn.Parameter(torch.zeros(heads * head_dim if concat else head_dim))
        if struct_dim is not None:
            self.att_src_s = nn.Parameter(torch.empty(heads, struct_dim))
            self.att_dst_s = nn.Parameter(torch.empty(heads, struct_dim))
            nn.init.xavier_uniform_(self.att_src_s)
            nn.init.xavier_uniform_(self.att_dst_s)
        self.dropout = nn.Dropout(dropout)
        self.att_dropout = nn.Dropout(att_dropout)
        self.negative_slope = negative_slope
        nn.init.xavier_uniform_(self.lin.weight)
        nn.init.xavier_uniform_(self.att_src)
        nn.init.xavier_uniform_(self.att_dst)

    def forward(self, x, edge_index, num_nodes, struct_embed=None, return_attn=False):
        row, col = edge_index
        x = self.dropout(x)
        h = self.lin(x).view(-1, self.heads, self.head_dim)
        alpha_src = (h * self.att_src).sum(dim=-1)
        alpha_dst = (h * self.att_dst).sum(dim=-1)
        if self.struct_dim is not None:
            alpha_src = alpha_src + (struct_embed.unsqueeze(1) * self.att_src_s).sum(dim=-1)
            alpha_dst = alpha_dst + (struct_embed.unsqueeze(1) * self.att_dst_s).sum(dim=-1)
        edge_alpha = F.leaky_relu(alpha_src[col] + alpha_dst[row], self.negative_slope)
        edge_alpha = torch.stack([scatter_softmax(edge_alpha[:, k], row, num_nodes) for k in range(self.heads)], dim=-1)
        edge_alpha_dropped = self.att_dropout(edge_alpha)
        msg = h[col] * edge_alpha_dropped.unsqueeze(-1)
        out = scatter_add(msg.reshape(msg.size(0), -1), row, num_nodes).view(num_nodes, self.heads, self.head_dim)
        out = (out.reshape(num_nodes, self.heads * self.head_dim) if self.concat else out.mean(dim=1)) + self.bias
        return (out, edge_alpha) if return_attn else out


class GAT(nn.Module):
    """Stack of GATLayers; selects the vanilla / structural-concat / SA-GAT variant via struct_dim."""
    def __init__(self, in_dim, head_dim, out_dim, num_layers, heads, out_heads, dropout, att_dropout, negative_slope, struct_dim=None, struct_embed_dim=16):
        super().__init__()
        self.use_struct_attn = struct_dim is not None
        sdim = None
        if self.use_struct_attn:
            self.struct_mlp = nn.Sequential(nn.Linear(struct_dim, struct_embed_dim), nn.ReLU(), nn.Dropout(dropout))
            sdim = struct_embed_dim
        self.layers = nn.ModuleList()
        dim = in_dim
        for _ in range(num_layers - 1):
            self.layers.append(GATLayer(dim, head_dim, heads, dropout, att_dropout, negative_slope, concat=True, struct_dim=sdim))
            dim = head_dim * heads
        self.layers.append(GATLayer(dim, out_dim, out_heads, dropout, att_dropout, negative_slope, concat=False, struct_dim=sdim))
        self.elu = nn.ELU()

    def forward(self, x, edge_index, num_nodes, struct_feats=None, return_attn=False):
        se = self.struct_mlp(struct_feats) if self.use_struct_attn else None
        attn_first = None
        for i, layer in enumerate(self.layers[:-1]):
            if return_attn and i == 0:
                x, attn_first = layer(x, edge_index, num_nodes, struct_embed=se, return_attn=True)
            else:
                x = layer(x, edge_index, num_nodes, struct_embed=se)
            x = self.elu(x)
        out = self.layers[-1](x, edge_index, num_nodes, struct_embed=se)
        return (out, attn_first) if return_attn else out

In [ ]:
def train_epoch_sagat(model, x, edge_index, num_nodes, labels, train_idx, optimizer, evaluator, struct_feats):
    """One full-batch training step for GAT/SA-GAT."""
    model.train()
    optimizer.zero_grad()
    out = model(x, edge_index, num_nodes, struct_feats=struct_feats)
    loss = F.cross_entropy(out[train_idx], labels[train_idx])
    loss.backward()
    optimizer.step()
    pred = out[train_idx].argmax(dim=-1).detach().cpu()
    acc = evaluator.eval({'y_true': labels[train_idx].cpu().view(-1, 1), 'y_pred': pred.view(-1, 1)})['acc']
    return float(loss.item()), acc


@torch.no_grad()
def evaluate_sagat(model, x, edge_index, num_nodes, labels, idx, evaluator, struct_feats):
    """Full-batch accuracy on a given node index set."""
    model.eval()
    out = model(x, edge_index, num_nodes, struct_feats=struct_feats)
    pred = out[idx].argmax(dim=-1).cpu()
    return evaluator.eval({'y_true': labels[idx].cpu().view(-1, 1), 'y_pred': pred.view(-1, 1)})['acc']


@torch.no_grad()
def analyze_structural_attention(model, x, edge_index, num_nodes, struct_feats, raw_struct, out_dir, logger):
    """Pearson-correlate layer-1 attention weights against each structural statistic of the neighbour."""
    model.eval()
    _, attn = model(x, edge_index, num_nodes, struct_feats=struct_feats, return_attn=True)
    if attn is None:
        return None
    alpha = attn.mean(dim=1).cpu().numpy()
    row, col = edge_index.cpu().numpy()
    mask = row != col
    alpha, src = alpha[mask], col[mask]
    correlations = {}
    for name in ('degree', 'pagerank', 'clustering', 'betweenness'):
        r, p = pearsonr(raw_struct[name][src], alpha)
        correlations[name] = {'r': float(r), 'p': float(p)}
        logger.info('attention vs %-11s : r=%+.4f (p=%.2e)', name, r, p)
    write_json(out_dir / 'attention_structure_correlation.json', correlations)
    return correlations


def run_sagat(args, run_id, device):
    """Full training loop for one SA-GAT variant (gat / gat-concat / sagat), plus the attention-structure analysis for 'sagat'."""
    run_name = f"{datetime.now().strftime('%Y%m%d_%H%M%S')}_run{run_id}_{args.variant}"
    out_dir = make_output_dir(OUTPUTS_ROOT / 'sagat', run_name)
    run_logger = setup_logger('sagat', out_dir)
    run_logger.info('Args: %s', json.dumps({k: v for k, v in vars(args).items()}, default=str, sort_keys=True))
    set_seed(args.seed + run_id)

    struct_feats_norm = raw_struct = None
    if args.variant in ('gat-concat', 'sagat'):
        struct_feats_norm, raw_struct = compute_structural_features(data.edge_index, data.num_nodes, args.betweenness_samples, args.seed, run_logger)

    x = data.x.float()
    if args.variant == 'gat-concat':
        x = torch.cat([x, struct_feats_norm], dim=-1)
    x = x.to(device)

    edge_index = add_self_loops(data.edge_index, data.num_nodes).to(device)
    labels_dev = labels.to(device)
    split_dev = {k: v.to(device) for k, v in split_idx.items()}
    struct_dev = struct_feats_norm.to(device) if args.variant == 'sagat' else None

    model = GAT(
        x.size(1), args.head_dim, num_classes, args.num_layers, args.heads, args.out_heads,
        args.dropout, args.att_dropout, args.negative_slope,
        struct_dim=(struct_feats_norm.size(1) if args.variant == 'sagat' else None),
        struct_embed_dim=args.struct_embed_dim,
    ).to(device)
    run_logger.info('Model params: %d', count_params(model))
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)

    best_val = best_test = -1.0
    best_epoch = stale = 0
    ckpt_path = out_dir / 'best.pt'
    metrics_path = out_dir / 'metrics.jsonl'

    for epoch in range(args.epochs):
        t0 = time.time()
        loss, train_acc = train_epoch_sagat(model, x, edge_index, data.num_nodes, labels_dev, split_dev['train'], optimizer, evaluator, struct_dev)
        val_acc = evaluate_sagat(model, x, edge_index, data.num_nodes, labels_dev, split_dev['valid'], evaluator, struct_dev)
        if val_acc > best_val:
            best_val, best_epoch = val_acc, epoch
            best_test = evaluate_sagat(model, x, edge_index, data.num_nodes, labels_dev, split_dev['test'], evaluator, struct_dev)
            torch.save({'model_state': model.state_dict()}, ckpt_path)
            stale = 0
        else:
            stale += 1
        append_jsonl(metrics_path, {
            'epoch': epoch, 'loss': loss, 'train_acc': train_acc, 'val_acc': val_acc,
            'best_val': best_val, 'best_test': best_test, 'best_epoch': best_epoch, 'time_sec': time.time() - t0,
        })
        if stale >= args.patience:
            run_logger.info('Early stopping at epoch=%d', epoch)
            break

    if ckpt_path.exists():
        model.load_state_dict(torch.load(ckpt_path, map_location=device)['model_state'])

    result = {'run': run_id, 'model_family': f'sagat/{args.variant}', 'variant': args.variant, 'num_params': count_params(model), 'best_val': best_val, 'best_test': best_test, 'best_epoch': best_epoch, 'output_dir': str(out_dir)}
    if args.variant == 'sagat' and raw_struct is not None and args.num_layers >= 2:
        correlations = analyze_structural_attention(model, x, edge_index, data.num_nodes, struct_dev, raw_struct, out_dir, run_logger)
        if correlations:
            result['attention_structure_correlation'] = correlations

    write_json(out_dir / 'results.json', result)
    plot_training_curves(metrics_path, out_dir, title=f"SA-GAT ({args.variant})")
    run_logger.info('Final: %s', result)
    print(f"sagat/{args.variant}: best_val={best_val:.4f}  best_test={best_test:.4f}")
    return result

In [ ]:
def sagat_args(variant):
    return SimpleNamespace(
        variant=variant, head_dim=32, heads=8, out_heads=2, num_layers=2, struct_embed_dim=16,
        dropout=0.6, att_dropout=0.6, negative_slope=0.2, betweenness_samples=500,
        epochs=epochs_for(200), lr=0.005, weight_decay=5e-4, patience=30, seed=0,
    )

res_gat = run_sagat(sagat_args('gat'), run_id=0, device=DEVICE)
results.append(res_gat)

In [ ]:
res_sagat = run_sagat(sagat_args('sagat'), run_id=0, device=DEVICE)
results.append(res_sagat)
res_sagat.get('attention_structure_correlation')

## 6. GAMLP

Decoupled propagation: K-hop mean-propagated features are precomputed once,
then a hop-attention MLP is trained on them. `mode='rlu'` additionally
propagates training labels through the graph and self-trains across stages on
high-confidence pseudo-labels — set below for the plain baseline by default.

In [ ]:
class Dense(nn.Module):
    """Linear + BatchNorm + residual-if-same-shape -- GAMLP's basic building block."""
    def __init__(self, in_features, out_features, use_bn=True):
        super().__init__()
        self.in_features, self.out_features = in_features, out_features
        self.weight = nn.Parameter(torch.empty(in_features, out_features))
        self.bias = nn.BatchNorm1d(out_features) if use_bn else nn.Identity()
        self.reset_parameters()

    def reset_parameters(self):
        stdv = 1.0 / (self.out_features ** 0.5)
        self.weight.data.uniform_(-stdv, stdv)
        if isinstance(self.bias, nn.BatchNorm1d):
            self.bias.reset_parameters()

    def forward(self, x):
        out = self.bias(torch.mm(x, self.weight))
        return out + x if self.in_features == self.out_features else out


class GraphConvolution(nn.Module):
    """Dense layer with an APPNP-style residual mix (1-alpha)*x + alpha*h0 before the linear transform."""
    def __init__(self, in_features, out_features, alpha, use_bn=True):
        super().__init__()
        self.in_features, self.out_features, self.alpha = in_features, out_features, alpha
        self.weight = nn.Parameter(torch.empty(in_features, out_features))
        self.bias = nn.BatchNorm1d(out_features) if use_bn else nn.Identity()
        self.reset_parameters()

    def reset_parameters(self):
        stdv = 1.0 / (self.out_features ** 0.5)
        self.weight.data.uniform_(-stdv, stdv)
        if isinstance(self.bias, nn.BatchNorm1d):
            self.bias.reset_parameters()

    def forward(self, x, h0):
        support = (1.0 - self.alpha) * x + self.alpha * h0
        out = self.bias(torch.mm(support, self.weight))
        return out + x if self.in_features == self.out_features else out


class FeedForwardNet(nn.Module):
    """Plain MLP with BatchNorm + PReLU + dropout between layers."""
    def __init__(self, in_feats, hidden, out_feats, n_layers, dropout, use_bn=True):
        super().__init__()
        self.layers, self.bns, self.n_layers, self.use_bn = nn.ModuleList(), nn.ModuleList(), n_layers, use_bn
        if n_layers == 1:
            self.layers.append(nn.Linear(in_feats, out_feats))
        else:
            self.layers.append(nn.Linear(in_feats, hidden))
            self.bns.append(nn.BatchNorm1d(hidden))
            for _ in range(n_layers - 2):
                self.layers.append(nn.Linear(hidden, hidden))
                self.bns.append(nn.BatchNorm1d(hidden))
            self.layers.append(nn.Linear(hidden, out_feats))
            self.prelu = nn.PReLU()
            self.dropout = nn.Dropout(dropout)
        self.reset_parameters()

    def reset_parameters(self):
        gain = nn.init.calculate_gain('relu')
        for layer in self.layers:
            nn.init.xavier_uniform_(layer.weight, gain=gain)
            nn.init.zeros_(layer.bias)
        for bn in self.bns:
            bn.reset_parameters()

    def forward(self, x):
        for layer_id, layer in enumerate(self.layers):
            x = layer(x)
            if layer_id < self.n_layers - 1:
                if self.use_bn:
                    x = self.bns[layer_id](x)
                x = self.dropout(self.prelu(x))
        return x


class FeedForwardNetII(nn.Module):
    """MLP built from Dense/GraphConvolution layers with a fixed skip connection to the first layer's output (h0)."""
    def __init__(self, in_feats, hidden, out_feats, n_layers, dropout, alpha, use_bn=True):
        super().__init__()
        self.layers, self.n_layers = nn.ModuleList(), n_layers
        if n_layers == 1:
            self.layers.append(Dense(in_feats, out_feats, use_bn=False))
        else:
            self.layers.append(Dense(in_feats, hidden, use_bn=use_bn))
            for _ in range(n_layers - 2):
                self.layers.append(GraphConvolution(hidden, hidden, alpha, use_bn=use_bn))
            self.layers.append(Dense(hidden, out_feats, use_bn=False))
        self.prelu = nn.PReLU()
        self.dropout = nn.Dropout(dropout)
        self.reset_parameters()

    def reset_parameters(self):
        for layer in self.layers:
            layer.reset_parameters()

    def forward(self, x):
        x = self.layers[0](x)
        h0 = x
        for layer_id, layer in enumerate(self.layers[1:], start=1):
            x = self.dropout(self.prelu(x))
            x = layer(x) if layer_id == self.n_layers - 1 else layer(x, h0)
        return x


def activation(name):
    """Map a name string to an nn.Module activation."""
    if name == 'sigmoid':
        return nn.Sigmoid()
    if name == 'leaky_relu':
        return nn.LeakyReLU(0.2)
    return nn.ReLU()

In [ ]:
class RGAMLP(nn.Module):
    """GAMLP variant: recursively-built attention over K hop features."""
    def __init__(self, nfeat, hidden, nclass, num_hops, args, use_label=False):
        super().__init__()
        self.num_hops, self.pre_process, self.residual, self.use_label = num_hops, args.pre_process, args.residual, use_label
        att_dim = hidden if self.pre_process else nfeat
        self.process = nn.ModuleList([FeedForwardNet(nfeat, hidden, hidden, 2, args.dropout, args.bns) for _ in range(num_hops)]) if self.pre_process else None
        self.lr_att = nn.Linear(att_dim + att_dim, 1)
        self.lr_output = FeedForwardNetII(att_dim, hidden, nclass, args.n_layers_2, args.dropout, args.alpha, args.bns)
        self.res_fc = nn.Linear(nfeat, att_dim)
        self.label_fc = FeedForwardNet(nclass, hidden, nclass, args.n_layers_3, args.dropout, args.bns) if use_label else None
        self.input_drop, self.att_drop, self.label_drop, self.dropout = nn.Dropout(args.input_drop), nn.Dropout(args.att_drop), nn.Dropout(args.label_drop), nn.Dropout(args.dropout)
        self.prelu, self.act = nn.PReLU(), activation(args.act)
        self.reset_parameters()

    def reset_parameters(self):
        gain = nn.init.calculate_gain('relu')
        nn.init.xavier_uniform_(self.lr_att.weight, gain=gain); nn.init.zeros_(self.lr_att.bias)
        nn.init.xavier_uniform_(self.res_fc.weight, gain=gain); nn.init.zeros_(self.res_fc.bias)
        self.lr_output.reset_parameters()
        if self.process is not None:
            for layer in self.process:
                layer.reset_parameters()
        if self.label_fc is not None:
            self.label_fc.reset_parameters()

    def encode(self, feature_list):
        feature_list = [self.input_drop(f) for f in feature_list]
        if self.pre_process:
            return [self.process[i](feature_list[i]) for i in range(self.num_hops)], feature_list
        return feature_list, feature_list

    def forward(self, feature_list, label_emb=None):
        input_list, raw_list = self.encode(feature_list)
        attention_scores = [self.act(self.lr_att(torch.cat([input_list[0], input_list[0]], dim=1)))]
        for i in range(1, self.num_hops):
            att = F.softmax(torch.cat(attention_scores[:i], dim=1), dim=1)
            history = input_list[0] * self.att_drop(att[:, 0:1])
            for j in range(1, i):
                history = history + input_list[j] * self.att_drop(att[:, j:j + 1])
            attention_scores.append(self.act(self.lr_att(torch.cat([history, input_list[i]], dim=1))))
        scores = F.softmax(torch.cat(attention_scores, dim=1), dim=1)
        hidden = input_list[0] * self.att_drop(scores[:, 0:1])
        for i in range(1, self.num_hops):
            hidden = hidden + input_list[i] * self.att_drop(scores[:, i:i + 1])
        if self.residual:
            hidden = self.dropout(self.prelu(hidden + self.res_fc(raw_list[0])))
        out = self.lr_output(hidden)
        if self.use_label and label_emb is not None:
            out = out + self.label_fc(self.label_drop(label_emb))
        return out


class JKGAMLP(nn.Module):
    """GAMLP variant: attention over K hop features driven by a Jumping-Knowledge reference vector."""
    def __init__(self, nfeat, hidden, nclass, num_hops, args, use_label=False):
        super().__init__()
        self.num_hops, self.pre_process, self.residual, self.use_label = num_hops, args.pre_process, args.residual, use_label
        att_dim = hidden if self.pre_process else nfeat
        self.process = nn.ModuleList([FeedForwardNet(nfeat, hidden, hidden, 2, args.dropout, args.bns) for _ in range(num_hops)]) if self.pre_process else None
        self.lr_jk_ref = FeedForwardNetII(num_hops * att_dim, hidden, hidden, args.n_layers_1, args.dropout, args.alpha, args.bns)
        self.lr_att = nn.Linear(att_dim + hidden, 1)
        self.lr_output = FeedForwardNetII(att_dim, hidden, nclass, args.n_layers_2, args.dropout, args.alpha, args.bns)
        self.res_fc = nn.Linear(nfeat, att_dim)
        self.label_fc = FeedForwardNet(nclass, hidden, nclass, args.n_layers_3, args.dropout, args.bns) if use_label else None
        self.input_drop, self.att_drop, self.label_drop, self.dropout = nn.Dropout(args.input_drop), nn.Dropout(args.att_drop), nn.Dropout(args.label_drop), nn.Dropout(args.dropout)
        self.prelu, self.act = nn.PReLU(), activation(args.act)
        self.reset_parameters()

    def reset_parameters(self):
        gain = nn.init.calculate_gain('relu')
        nn.init.xavier_uniform_(self.lr_att.weight, gain=gain); nn.init.zeros_(self.lr_att.bias)
        nn.init.xavier_uniform_(self.res_fc.weight, gain=gain); nn.init.zeros_(self.res_fc.bias)
        self.lr_jk_ref.reset_parameters(); self.lr_output.reset_parameters()
        if self.process is not None:
            for layer in self.process:
                layer.reset_parameters()
        if self.label_fc is not None:
            self.label_fc.reset_parameters()

    def forward(self, feature_list, label_emb=None):
        feature_list = [self.input_drop(f) for f in feature_list]
        input_list = [self.process[i](feature_list[i]) for i in range(self.num_hops)] if self.pre_process else feature_list
        jk_ref = self.dropout(self.prelu(self.lr_jk_ref(torch.cat(input_list, dim=1))))
        scores = [self.act(self.lr_att(torch.cat([jk_ref, x], dim=1))) for x in input_list]
        weights = F.softmax(torch.cat(scores, dim=1), dim=1)
        hidden = input_list[0] * self.att_drop(weights[:, 0:1])
        for i in range(1, self.num_hops):
            hidden = hidden + input_list[i] * self.att_drop(weights[:, i:i + 1])
        if self.residual:
            hidden = self.dropout(self.prelu(hidden + self.res_fc(feature_list[0])))
        out = self.lr_output(hidden)
        if self.use_label and label_emb is not None:
            out = out + self.label_fc(self.label_drop(label_emb))
        return out


def build_gamlp(args, in_feats, num_classes, use_label):
    """Construct an RGAMLP or JKGAMLP model depending on args.method."""
    num_hops = args.num_hops + 1
    cls = JKGAMLP if args.method == 'JK_GAMLP' else RGAMLP
    return cls(in_feats, args.hidden, num_classes, num_hops, args, use_label=use_label)

In [ ]:
@torch.no_grad()
def precompute_hop_features(num_hops, cache_dir, cache_features, logger):
    """Precompute and cache X, AX, A^2 X, ..., A^K X (mean-propagated hop features)."""
    cache_path = Path(cache_dir) / f'amazon_computer_hops_{num_hops}.pt'
    if cache_features and cache_path.exists():
        logger.info('Loading cached hop features from %s', cache_path)
        return torch.load(cache_path, map_location='cpu')
    logger.info('Precomputing %d-hop mean features', num_hops)
    row, col = data.edge_index
    deg = scatter_add(row.new_ones(row.size(0), 1, dtype=torch.float32), row, data.num_nodes).clamp(min=1)
    feats = [data.x.float().cpu()]
    for hop in range(1, num_hops + 1):
        t0 = time.time()
        feats.append(propagate_mean(data.edge_index, data.num_nodes, feats[-1], deg=deg, hops=1))
        logger.info('Computed hop %d feature in %.2fs', hop, time.time() - t0)
        gc.collect()
    if cache_features:
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        torch.save(feats, cache_path)
    return feats


@torch.no_grad()
def prepare_label_emb(deg, split_idx_local, num_classes, num_hops, teacher_probs=None):
    """Propagate (possibly pseudo-labelled) training labels through the graph for RLU mode."""
    y = torch.zeros(labels.size(0), num_classes, dtype=torch.float32)
    train_idx = split_idx_local['train']
    y[train_idx] = F.one_hot(labels[train_idx], num_classes=num_classes).float()
    if teacher_probs is not None:
        y[split_idx_local['valid']] = teacher_probs[split_idx_local['valid']]
        y[split_idx_local['test']] = teacher_probs[split_idx_local['test']]
    return propagate_mean(data.edge_index, data.num_nodes, y, deg=deg, hops=num_hops)


def ogb_acc(evaluator, y_true, y_pred):
    """Accuracy via the shared AccuracyEvaluator, OGB-style {y_true, y_pred} interface."""
    return evaluator.eval({'y_true': y_true.view(-1, 1), 'y_pred': y_pred.view(-1, 1)})['acc']


def run_batches(indices, batch_size, shuffle):
    """Wrap a node index tensor in a DataLoader for mini-batching."""
    return torch.utils.data.DataLoader(indices.cpu(), batch_size=batch_size, shuffle=shuffle, drop_last=False)

In [ ]:
def train_epoch_gamlp(model, feats, labels, label_emb, train_idx, optimizer, evaluator, batch_size, device):
    """One mini-batch training epoch in plain mode (hard labels only)."""
    model.train()
    total_loss = total_n = 0
    y_true, y_pred = [], []
    for batch in run_batches(train_idx, batch_size, shuffle=True):
        batch_feats = [f[batch].to(device) for f in feats]
        batch_label_emb = label_emb[batch].to(device) if label_emb is not None else None
        out = model(batch_feats, batch_label_emb)
        y = labels[batch].to(device)
        loss = F.cross_entropy(out, y)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total_loss += float(loss.item()) * batch.numel(); total_n += batch.numel()
        y_true.append(y.detach().cpu()); y_pred.append(out.argmax(dim=-1).detach().cpu())
    return total_loss / max(total_n, 1), ogb_acc(evaluator, torch.cat(y_true), torch.cat(y_pred))


def train_epoch_rlu_gamlp(model, feats, labels, label_emb, train_idx, enhance_idx, teacher_probs, optimizer, evaluator, args, device):
    """One mini-batch training epoch in RLU mode: hard loss on train nodes + KL consistency loss on confident pseudo-labelled nodes."""
    model.train()
    n = max(len(train_idx) + len(enhance_idx), 1)
    train_loader = run_batches(train_idx, max(1, int(args.batch_size * len(train_idx) / n)), True)
    enhance_loader = run_batches(enhance_idx, max(1, int(args.batch_size * len(enhance_idx) / n)), True)
    loss_sum = total_n = 0
    y_true, y_pred = [], []
    for idx_1, idx_2 in zip(train_loader, enhance_loader):
        idx = torch.cat([idx_1, idx_2], dim=0)
        batch_feats = [f[idx].to(device) for f in feats]
        out = model(batch_feats, label_emb[idx].to(device))
        hard_loss = F.cross_entropy(out[:idx_1.numel()], labels[idx_1].to(device))
        teacher_soft = teacher_probs[idx_2].to(device)
        teacher_conf = teacher_soft.max(dim=1, keepdim=True).values
        kl_loss = (teacher_conf * (teacher_soft * (torch.log(teacher_soft + 1e-8) - F.log_softmax(out[idx_1.numel():], dim=1))).sum(dim=1, keepdim=True)).mean()
        ratio_hard, ratio_soft = idx_1.numel() / max(idx.numel(), 1), idx_2.numel() / max(idx.numel(), 1)
        loss = hard_loss * ratio_hard + kl_loss * ratio_soft * args.gama
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        loss_sum += float(loss.item()) * idx.numel(); total_n += idx.numel()
        y_true.append(labels[idx_1].cpu()); y_pred.append(out[:idx_1.numel()].argmax(dim=-1).detach().cpu())
    return loss_sum / max(total_n, 1), ogb_acc(evaluator, torch.cat(y_true), torch.cat(y_pred))


@torch.no_grad()
def evaluate_gamlp(model, feats, labels, label_emb, idx, evaluator, batch_size, device):
    """Mini-batch accuracy on a given node index set."""
    model.eval()
    preds = []
    for batch in run_batches(idx, batch_size, shuffle=False):
        batch_feats = [f[batch].to(device) for f in feats]
        batch_label_emb = label_emb[batch].to(device) if label_emb is not None else None
        preds.append(model(batch_feats, batch_label_emb).argmax(dim=-1).cpu())
    return ogb_acc(evaluator, labels[idx.cpu()], torch.cat(preds))


@torch.no_grad()
def predict_logits_gamlp(model, feats, label_emb, batch_size, device):
    """Mini-batch forward pass over every node, returning logits."""
    model.eval()
    logits = []
    for batch in run_batches(torch.arange(feats[0].size(0)), batch_size, shuffle=False):
        batch_feats = [f[batch].to(device) for f in feats]
        batch_label_emb = label_emb[batch].to(device) if label_emb is not None else None
        logits.append(model(batch_feats, batch_label_emb).cpu())
    return torch.cat(logits, dim=0)


def train_stage_gamlp(args, stage, epochs, model, feats, label_emb, split_idx_local, evaluator, out_dir, run_logger, device, teacher_probs=None):
    """Train one RLU stage to convergence (early stopping), building the enhance-set from confident pseudo-labels first."""
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    best_val = best_test = -1.0
    best_epoch = stale = 0
    train_idx = split_idx_local['train']
    enhance_idx = None
    if args.mode == 'rlu' and stage > 0 and teacher_probs is not None:
        conf = teacher_probs.max(dim=1).values
        mask = conf > args.threshold
        mask[train_idx] = False
        enhance_idx = torch.nonzero(mask, as_tuple=False).view(-1).cpu()
        run_logger.info('Stage %d confident extra nodes: %d', stage, enhance_idx.numel())
        if enhance_idx.numel() == 0:
            enhance_idx = train_idx.cpu()

    ckpt_path = out_dir / f'best_stage_{stage}.pt'
    metrics_path = out_dir / 'metrics.jsonl'
    for epoch in range(epochs):
        t0 = time.time()
        if args.mode == 'rlu' and stage > 0 and teacher_probs is not None:
            loss, train_acc = train_epoch_rlu_gamlp(model, feats, labels, label_emb, train_idx, enhance_idx, teacher_probs, optimizer, evaluator, args, device)
        else:
            loss, train_acc = train_epoch_gamlp(model, feats, labels, label_emb, train_idx, optimizer, evaluator, args.batch_size, device)
        val_acc = evaluate_gamlp(model, feats, labels, label_emb, split_idx_local['valid'], evaluator, args.batch_size, device)
        if val_acc > best_val:
            best_val, best_epoch = val_acc, epoch
            best_test = evaluate_gamlp(model, feats, labels, label_emb, split_idx_local['test'], evaluator, args.batch_size, device)
            torch.save({'model_state': model.state_dict()}, ckpt_path)
            stale = 0
        else:
            stale += 1
        append_jsonl(metrics_path, {
            'stage': stage, 'epoch': epoch, 'loss': loss, 'train_acc': train_acc, 'val_acc': val_acc,
            'best_val': best_val, 'best_test': best_test, 'best_epoch': best_epoch, 'time_sec': time.time() - t0,
        })
        if stale >= args.patience:
            run_logger.info('Early stopping at stage=%d epoch=%d', stage, epoch)
            break
    if ckpt_path.exists():
        model.load_state_dict(torch.load(ckpt_path, map_location=device)['model_state'])
    return best_val, best_test, best_epoch, ckpt_path


def run_gamlp(args, run_id, device):
    """Full training loop across all stages (1 stage in 'plain' mode, several in 'rlu' self-training mode)."""
    run_name = f"{datetime.now().strftime('%Y%m%d_%H%M%S')}_run{run_id}_{args.mode}_{args.method}"
    out_dir = make_output_dir(OUTPUTS_ROOT / 'gamlp', run_name)
    run_logger = setup_logger('gamlp', out_dir)
    run_logger.info('Args: %s', json.dumps(vars(args), sort_keys=True, default=str))
    set_seed(args.seed + run_id)

    feats = precompute_hop_features(args.num_hops, args.cache_dir, args.cache_features, run_logger)
    row, col = data.edge_index
    deg = scatter_add(row.new_ones(row.size(0), 1, dtype=torch.float32), row, data.num_nodes).clamp(min=1)
    split_local = {k: v.cpu() for k, v in split_idx.items()}

    teacher_probs = None
    stage_results = []
    stages = args.stages if args.mode == 'rlu' else [args.epochs]
    for stage, stage_epochs in enumerate(stages):
        label_emb = None
        use_label = args.mode == 'rlu'
        if use_label:
            label_emb = prepare_label_emb(deg, split_local, num_classes, args.label_num_hops, teacher_probs)
        model = build_gamlp(args, feats[0].size(1), num_classes, use_label=use_label).to(device)
        run_logger.info('Stage %d model params: %d', stage, count_params(model))
        best_val, best_test, best_epoch, ckpt_path = train_stage_gamlp(args, stage, stage_epochs, model, feats, label_emb, split_local, evaluator, out_dir, run_logger, device, teacher_probs)
        logits = predict_logits_gamlp(model, feats, label_emb, args.batch_size, device)
        teacher_probs = (logits / args.temp).softmax(dim=1)
        stage_results.append({'stage': stage, 'best_val': best_val, 'best_test': best_test, 'best_epoch': best_epoch})
        gc.collect()

    result = {
        'run': run_id, 'model_family': 'gamlp', 'mode': args.mode, 'method': args.method,
        'num_params': count_params(model),  # last stage's model
        'best_val': stage_results[-1]['best_val'], 'best_test': stage_results[-1]['best_test'],
        'best_epoch': stage_results[-1]['best_epoch'], 'stages': stage_results, 'output_dir': str(out_dir),
    }
    write_json(out_dir / 'results.json', result)
    plot_training_curves(out_dir / 'metrics.jsonl', out_dir, title=f"GAMLP ({args.mode})")
    run_logger.info('Final result: %s', result)
    print(f"gamlp/{args.mode}: best_val={result['best_val']:.4f}  best_test={result['best_test']:.4f}")
    return result

In [ ]:
args_gamlp = SimpleNamespace(
    mode='plain', method='R_GAMLP',
    num_hops=5, label_num_hops=9, hidden=512, n_layers_1=4, n_layers_2=4, n_layers_3=4,
    dropout=0.5, input_drop=0.2, att_drop=0.5, label_drop=0.0, alpha=0.5, act='leaky_relu',
    pre_process=True, residual=True, bns=True,
    epochs=epochs_for(300), stages=[epochs_for(400), epochs_for(300), epochs_for(300), epochs_for(300)],
    batch_size=50000, lr=0.001, weight_decay=0.0, patience=100,
    threshold=0.85, temp=1.0, gama=0.1,
    cache_features=True, cache_dir=str(OUTPUTS_ROOT / 'cache'), seed=0,
)
res_gamlp = run_gamlp(args_gamlp, run_id=0, device=DEVICE)
results.append(res_gamlp)

## 7. Transformer GAMLP

A lean, plain-only GAMLP variant that replaces the usual hop-attention MLP
with a Transformer over hop tokens:

    multi-operator propagation -> multi-hop tokens -> Transformer encoder
    -> attention pooling -> gated-residual classifier

"Multi-operator" means each hop is precomputed with *two* normalisations of
the same K-hop propagation instead of one -- mean (`D^-1 A`, GAMLP's usual
choice) and symmetric (`D^-1/2 A D^-1/2`, GCN's normalisation) -- giving
`2K + 1` tokens total. Feeding both as separate tokens lets attention pooling
pick whichever normalisation is more informative per node, rather than
committing to one propagation rule for every node. No RLU / label
propagation / self-training here, unlike the plain GAMLP section above --
just a single supervised run.

Reuses `train_epoch_gamlp` / `evaluate_gamlp` / `predict_logits_gamlp` /
`run_batches` from the GAMLP section above unchanged: `TransformerGAMLP`'s
`forward(feature_list, label_emb=None)` has the exact same signature as
`RGAMLP` / `JKGAMLP`, it just ignores `label_emb`.

In [ ]:
@torch.no_grad()
def precompute_multi_operator_features(num_hops, cache_dir, cache_features, logger):
    """Precompute mean- and symmetric-normalised hop features (the token set below)."""
    cache_path = Path(cache_dir) / f'amazon_computer_multi_operator_hops_{num_hops}.pt'
    if cache_features and cache_path.exists():
        cached = torch.load(cache_path, map_location='cpu')
        logger.info('Loading cached multi-operator features from %s', cache_path)
        return cached['features'], cached['token_names']

    row, col = data.edge_index
    deg = scatter_add(row.new_ones(row.size(0), 1, dtype=torch.float32), row, data.num_nodes).clamp(min=1)
    sym_weight = gcn_norm_edge_weight(data.edge_index, data.num_nodes)  # D^-1/2 A D^-1/2, no self-loops

    raw = data.x.float().cpu()
    mean_features = [raw]
    symmetric_features = []
    symmetric = raw
    for hop in range(1, num_hops + 1):
        t0 = time.time()
        mean_features.append(propagate_mean(data.edge_index, data.num_nodes, mean_features[-1], deg=deg, hops=1))
        symmetric = weighted_propagate(symmetric, data.edge_index, sym_weight, data.num_nodes)
        symmetric_features.append(symmetric)
        logger.info('Computed multi-operator hop %d in %.2fs', hop, time.time() - t0)

    features = mean_features + symmetric_features
    token_names = [f'mean_hop_{h}' for h in range(num_hops + 1)] + [f'sym_hop_{h}' for h in range(1, num_hops + 1)]
    if cache_features:
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        torch.save({'features': features, 'token_names': token_names}, cache_path)
    return features, token_names


class GatedResidualBlock(nn.Module):
    """GELU-gated feed-forward block with a residual connection (the classifier's building block)."""

    def __init__(self, hidden, dropout):
        super().__init__()
        self.norm = nn.LayerNorm(hidden)
        self.expand = nn.Linear(hidden, 2 * hidden)
        self.project = nn.Linear(hidden, hidden)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        gate, value = self.expand(self.norm(x)).chunk(2, dim=-1)
        return x + self.dropout(self.project(F.gelu(gate) * value))


class TransformerGAMLP(nn.Module):
    """Multi-hop token Transformer with attention pooling and a gated-residual classifier."""

    def __init__(self, in_feats, hidden, nclass, num_tokens, args):
        super().__init__()
        if hidden % args.num_heads:
            raise ValueError('--hidden must be divisible by --num-heads')
        self.num_tokens = num_tokens
        self.input_drop = nn.Dropout(args.input_drop)
        self.hop_drop_prob = args.hop_dropout
        self.token_projection = nn.Sequential(nn.Linear(in_feats, hidden), nn.GELU(), nn.LayerNorm(hidden))
        self.token_position = nn.Parameter(torch.empty(1, num_tokens, hidden))
        if args.token_layers:
            layer = nn.TransformerEncoderLayer(
                d_model=hidden, nhead=args.num_heads, dim_feedforward=hidden * args.ffn_multiplier,
                dropout=args.attention_dropout, activation='gelu', batch_first=True, norm_first=True,
            )
            self.token_encoder = nn.TransformerEncoder(layer, num_layers=args.token_layers, norm=nn.LayerNorm(hidden))
        else:
            self.token_encoder = nn.Identity()
        self.pool_score = nn.Sequential(nn.LayerNorm(hidden), nn.Linear(hidden, hidden), nn.GELU(), nn.Linear(hidden, 1))
        self.classifier_input = nn.Sequential(nn.LayerNorm(hidden), nn.Linear(hidden, hidden), nn.GELU())
        self.classifier = nn.ModuleList(GatedResidualBlock(hidden, args.dropout) for _ in range(args.classifier_layers))
        self.output_norm = nn.LayerNorm(hidden)
        self.output = nn.Linear(hidden, nclass)
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.normal_(self.token_position, std=0.02)
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.LayerNorm):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)

    def forward(self, feature_list, label_emb=None):
        """Classify multi-hop feature tokens; label_emb is unused (plain mode only)."""
        tokens = torch.stack([self.input_drop(feature) for feature in feature_list], dim=1)
        tokens = self.token_projection(tokens) + self.token_position
        if self.training and self.hop_drop_prob:
            keep = torch.rand(tokens.shape[:2], device=tokens.device) >= self.hop_drop_prob
            keep[:, 0] = True
            tokens = tokens * keep.unsqueeze(-1)
        tokens = self.token_encoder(tokens)
        weights = F.softmax(self.pool_score(tokens).squeeze(-1), dim=1)
        fused = torch.sum(tokens * weights.unsqueeze(-1), dim=1)
        hidden = self.classifier_input(fused + tokens[:, 0])
        for block in self.classifier:
            hidden = block(hidden)
        return self.output(self.output_norm(hidden))

In [ ]:
def run_transformer_gamlp(args, run_id, device):
    """Full training loop for Transformer GAMLP (single stage, plain mode only)."""
    run_name = f"{datetime.now().strftime('%Y%m%d_%H%M%S')}_run{run_id}_transformer_gamlp"
    out_dir = make_output_dir(OUTPUTS_ROOT / 'transformer_gamlp', run_name)
    run_logger = setup_logger('transformer_gamlp', out_dir)
    run_logger.info('Args: %s', json.dumps(vars(args), sort_keys=True, default=str))
    set_seed(args.seed + run_id)

    features, token_names = precompute_multi_operator_features(args.num_hops, args.cache_dir, args.cache_features, run_logger)
    split_local = {k: v.cpu() for k, v in split_idx.items()}

    model = TransformerGAMLP(features[0].size(1), args.hidden, num_classes, len(features), args).to(device)
    run_logger.info('Using tokens: %s', token_names)
    run_logger.info('Model params: %d', count_params(model))
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)

    best_val = best_test = -1.0
    best_epoch = stale = 0
    ckpt_path = out_dir / 'best_model.pt'
    metrics_path = out_dir / 'metrics.jsonl'
    for epoch in range(args.epochs):
        t0 = time.time()
        loss, train_acc = train_epoch_gamlp(model, features, labels, None, split_local['train'], optimizer, evaluator, args.batch_size, device)
        val_acc = evaluate_gamlp(model, features, labels, None, split_local['valid'], evaluator, args.batch_size, device)
        if val_acc > best_val:
            best_val, best_epoch = val_acc, epoch
            best_test = evaluate_gamlp(model, features, labels, None, split_local['test'], evaluator, args.batch_size, device)
            torch.save({'model_state': model.state_dict()}, ckpt_path)
            stale = 0
        else:
            stale += 1
        append_jsonl(metrics_path, {
            'epoch': epoch, 'loss': loss, 'train_acc': train_acc, 'val_acc': val_acc,
            'best_val': best_val, 'best_test': best_test, 'best_epoch': best_epoch, 'time_sec': time.time() - t0,
        })
        if stale >= args.patience:
            run_logger.info('Early stopping at epoch=%d', epoch)
            break

    if ckpt_path.exists():
        model.load_state_dict(torch.load(ckpt_path, map_location=device)['model_state'])

    result = {
        'run': run_id, 'model_family': 'transformer_gamlp', 'feature_tokens': token_names,
        'num_params': count_params(model), 'best_val': best_val, 'best_test': best_test,
        'best_epoch': best_epoch, 'output_dir': str(out_dir),
    }
    write_json(out_dir / 'results.json', result)
    plot_training_curves(metrics_path, out_dir, title='Transformer GAMLP')
    run_logger.info('Final result: %s', result)
    print(f"transformer_gamlp: best_val={best_val:.4f}  best_test={best_test:.4f}")
    return result

In [ ]:
args_transformer_gamlp = SimpleNamespace(
    num_hops=5, hidden=256, token_layers=2, num_heads=8, ffn_multiplier=2, classifier_layers=2,
    dropout=0.35, input_drop=0.15, attention_dropout=0.1, hop_dropout=0.1,
    epochs=epochs_for(300), batch_size=50000, lr=0.001, weight_decay=1e-4, patience=100,
    cache_dir=str(OUTPUTS_ROOT / 'cache'), cache_features=True, seed=0,
)
res_transformer_gamlp = run_transformer_gamlp(args_transformer_gamlp, run_id=0, device=DEVICE)
results.append(res_transformer_gamlp)

## 8. LightGCN-GRAND

Combines two ideas: **LightGCN** propagation (no per-layer transform/nonlinearity
between hops — just a learned weighted sum of K propagated versions of the
logits) trained with **GRAND**'s DropNode augmentation + prediction-consistency
regularisation. An optional class-Compatibility-Propagation (CoP) channel
(`use_H=True` below) reweights the propagated logits using a learned
class-compatibility matrix seeded from an SGC + logistic-regression teacher's
pseudo-labels.

Runs on the same fixed `split_idx.csv` split as every other model here by
default (`low_label_split=False`) for a fair comparison; GRAND's consistency
regularisation was originally designed for a much smaller labelled set (20
train / 30 valid per class) — see `low_label_split` in the args cell below to
reproduce that regime instead (not comparable to the other rows in the
summary table if you do).

In [ ]:
class LightGCNGRAND(nn.Module):
    """LightGCN layer-combination propagation trained with GRAND-style regularisation; optional CoP compatibility channel."""
    def __init__(self, in_dim, hidden, num_classes, K, learn_gamma=True, use_H=True, Kc=2, H_init=None):
        super().__init__()
        self.K, self.use_H, self.Kc = K, use_H, Kc
        self.lin1 = nn.Linear(in_dim, hidden)
        self.lin2 = nn.Linear(hidden, num_classes)
        self.gamma = nn.Parameter(torch.zeros(K + 1), requires_grad=learn_gamma)
        if use_H:
            self.H = nn.Parameter(H_init.clone())
            self.delta = nn.Parameter(torch.zeros(Kc))
            self.mix = nn.Parameter(torch.tensor(-2.0))  # sigmoid(-2) ~ 0.12: CoP starts as a small correction

    def propagate(self, x, edge_index, edge_weight, num_nodes):
        g = F.softmax(self.gamma, dim=0)
        ego = x
        out = g[0] * ego
        for k in range(1, self.K + 1):
            ego = weighted_propagate(ego, edge_index, edge_weight, num_nodes)
            out = out + g[k] * ego
        return out

    def forward(self, x, edge_index, edge_weight, num_nodes, training, dropnode, dropout, prop_space):
        if training and dropnode > 0:
            mask = (torch.rand(num_nodes, 1, device=x.device) > dropnode).float() / (1.0 - dropnode)
            x = x * mask
        h = F.relu(self.lin1(x))
        if training:
            h = F.dropout(h, dropout, training=True)
        if prop_space == 'hidden':
            h = self.propagate(h, edge_index, edge_weight, num_nodes)
            return self.lin2(h)
        logits0 = self.lin2(h)
        zA = self.propagate(logits0, edge_index, edge_weight, num_nodes)
        if not self.use_H:
            return zA
        Hn = F.softmax(self.H, dim=1)
        P = F.softmax(logits0, dim=1)
        dl = F.softmax(self.delta, dim=0)
        ego, zB = P, torch.zeros_like(P)
        for k in range(self.Kc):
            ego = weighted_propagate(ego @ Hn, edge_index, edge_weight, num_nodes)
            zB = zB + dl[k] * ego
        m = torch.sigmoid(self.mix)
        return zA + m * torch.log(zB + 1e-8)


def build_normalized_adjacency_scipy(edge_index, num_nodes):
    """Scipy D^-1/2 (A+I) D^-1/2 normalised adjacency, used only by the numpy/sklearn teacher below."""
    row, col = edge_index.numpy()
    adj = sp.csr_matrix((np.ones(row.shape[0], dtype=np.float32), (row, col)), shape=(num_nodes, num_nodes))
    adj = adj.maximum(adj.T)
    adj.data[:] = 1
    m = adj + sp.eye(num_nodes)
    deg = np.asarray(m.sum(1)).ravel()
    d_inv_sqrt = sp.diags(deg ** -0.5)
    return (d_inv_sqrt @ m @ d_inv_sqrt).tocsr()


def teacher_h_init(x_np, labels_np, train_idx, edge_index, num_classes, mn_sp, seed):
    """Fit an SGC + logistic-regression teacher, pseudo-label every node, and build a class-compatibility matrix from label co-occurrence over edges."""
    hk = x_np.copy()
    acc = x_np.copy()
    for _ in range(2):
        hk = mn_sp @ hk
        acc = acc + hk
    feat_teacher = acc / 3.0
    clf = LogisticRegression(max_iter=1500, C=10.0, random_state=seed).fit(feat_teacher[train_idx], labels_np[train_idx])
    pseudo_labels = clf.predict(feat_teacher)

    row, col = edge_index.numpy()
    unique_edges = row < col
    e_row, e_col = row[unique_edges], col[unique_edges]
    pairs = np.zeros((num_classes, num_classes), dtype=np.float64)
    np.add.at(pairs, (pseudo_labels[e_row], pseudo_labels[e_col]), 1.0)
    pairs = pairs + pairs.T
    pairs = pairs / pairs.sum(1, keepdims=True).clip(min=1e-8)
    return torch.from_numpy(pairs.astype(np.float32))


def make_low_label_split(labels_np, num_classes, seed, n_train, n_valid):
    """GRAND-paper-style split: N train / M valid per class, rest test (independent of the benchmark's shared split_idx.csv)."""
    rng = np.random.RandomState(seed)
    train_ids, valid_ids, test_ids = [], [], []
    for c in range(num_classes):
        idx = rng.permutation(np.where(labels_np == c)[0])
        train_ids += list(idx[:n_train])
        valid_ids += list(idx[n_train:n_train + n_valid])
        test_ids += list(idx[n_train + n_valid:])
    return {
        'train': torch.tensor(train_ids, dtype=torch.long),
        'valid': torch.tensor(valid_ids, dtype=torch.long),
        'test': torch.tensor(test_ids, dtype=torch.long),
    }

In [ ]:
def forward_pass_lightgcn(model, x, edge_index, edge_weight, num_nodes, training, args):
    """Run one forward pass of the LightGCN-GRAND model with the args-derived hyperparameters."""
    return model(x, edge_index, edge_weight, num_nodes, training=training,
                 dropnode=args.dropnode, dropout=args.dropout, prop_space=args.prop_space)


def train_epoch_lightgcn(model, x, edge_index, edge_weight, num_nodes, labels, train_idx, optimizer, evaluator, lam, args):
    """One training step: S DropNode-augmented forward passes, supervised + GRAND consistency loss, backward."""
    model.train()
    probs = []
    for _ in range(args.S):
        logits = forward_pass_lightgcn(model, x, edge_index, edge_weight, num_nodes, True, args)
        probs.append(F.softmax(logits, dim=1))

    sup = sum(F.nll_loss(torch.log(pr[train_idx] + 1e-8), labels[train_idx]) for pr in probs) / args.S

    avg = sum(probs) / args.S
    sharp = avg.pow(1.0 / args.temp)
    sharp = (sharp / sharp.sum(dim=1, keepdim=True)).detach()
    conf_mask = (avg.max(dim=1).values > args.conf).float().detach()
    denom = conf_mask.sum() + 1e-8
    con = sum((conf_mask * (pr - sharp).pow(2).sum(dim=1)).sum() / denom for pr in probs) / args.S

    l2 = args.l2_reg * 0.5 * (model.lin1.weight.pow(2).sum() + model.lin2.weight.pow(2).sum())
    loss = sup + lam * con + l2

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    with torch.no_grad():
        pred = avg[train_idx].argmax(dim=1).cpu()
        acc = evaluator.eval({'y_true': labels[train_idx].cpu().view(-1, 1), 'y_pred': pred.view(-1, 1)})['acc']
    return float(loss.item()), acc


@torch.no_grad()
def evaluate_lightgcn(model, x, edge_index, edge_weight, num_nodes, labels, idx, evaluator, args):
    """Single forward pass (no DropNode) + accuracy on a given node index set."""
    model.eval()
    logits = forward_pass_lightgcn(model, x, edge_index, edge_weight, num_nodes, False, args)
    pred = logits[idx].argmax(dim=1).cpu()
    return evaluator.eval({'y_true': labels[idx].cpu().view(-1, 1), 'y_pred': pred.view(-1, 1)})['acc']


def run_lightgcn_grand(args, run_id, device):
    """Full training loop for LightGCN-GRAND, with lambda warm-up and CoP-channel teacher initialisation."""
    run_name = f"{datetime.now().strftime('%Y%m%d_%H%M%S')}_run{run_id}_lightgcn_grand"
    out_dir = make_output_dir(OUTPUTS_ROOT / 'lightgcn_grand', run_name)
    run_logger = setup_logger('lightgcn_grand', out_dir)
    run_logger.info('Args: %s', json.dumps(vars(args), sort_keys=True, default=str))
    set_seed(args.seed + run_id)

    split_local = split_idx
    if args.low_label_split:
        split_local = make_low_label_split(labels.numpy(), num_classes, args.seed + run_id, args.low_label_train_per_class, args.low_label_valid_per_class)
        run_logger.info('Low-label split (paper-style)  train=%d valid=%d test=%d', split_local['train'].numel(), split_local['valid'].numel(), split_local['test'].numel())

    x = data.x.float()
    edge_index_sl = add_self_loops(data.edge_index, data.num_nodes)
    edge_weight = gcn_norm_edge_weight(edge_index_sl, data.num_nodes)

    H_init = None
    if args.use_H:
        mn_sp = build_normalized_adjacency_scipy(data.edge_index, data.num_nodes)
        H_init = teacher_h_init(x.numpy(), labels.numpy(), split_local['train'].numpy(), data.edge_index, num_classes, mn_sp, args.seed + run_id).to(device)

    x, edge_index_sl, edge_weight = x.to(device), edge_index_sl.to(device), edge_weight.to(device)
    labels_dev = labels.to(device)
    split_dev = {k: v.to(device) for k, v in split_local.items()}

    model = LightGCNGRAND(x.size(1), args.hidden, num_classes, args.K, learn_gamma=args.learn_gamma, use_H=args.use_H, Kc=args.Kc, H_init=H_init).to(device)
    run_logger.info('Model params: %d', count_params(model))
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)

    best_val = best_test = -1.0
    best_epoch = stale = 0
    ckpt_path = out_dir / 'best.pt'
    metrics_path = out_dir / 'metrics.jsonl'

    for epoch in range(args.epochs):
        t0 = time.time()
        lam = args.lam * min(1.0, epoch / args.warmup) if args.warmup > 0 else args.lam
        loss, train_acc = train_epoch_lightgcn(model, x, edge_index_sl, edge_weight, data.num_nodes, labels_dev, split_dev['train'], optimizer, evaluator, lam, args)
        val_acc = evaluate_lightgcn(model, x, edge_index_sl, edge_weight, data.num_nodes, labels_dev, split_dev['valid'], evaluator, args)
        if val_acc > best_val:
            best_val, best_epoch = val_acc, epoch
            best_test = evaluate_lightgcn(model, x, edge_index_sl, edge_weight, data.num_nodes, labels_dev, split_dev['test'], evaluator, args)
            torch.save({'model_state': model.state_dict()}, ckpt_path)
            stale = 0
        else:
            stale += 1
        append_jsonl(metrics_path, {
            'epoch': epoch, 'loss': loss, 'train_acc': train_acc, 'val_acc': val_acc,
            'best_val': best_val, 'best_test': best_test, 'best_epoch': best_epoch, 'lambda': lam, 'time_sec': time.time() - t0,
        })
        if epoch >= args.min_epochs and stale >= args.patience:
            run_logger.info('Early stopping at epoch=%d', epoch)
            break

    result = {
        'run': run_id, 'model_family': 'lightgcn_grand', 'prop_space': args.prop_space, 'use_H': args.use_H,
        'num_params': count_params(model), 'best_val': best_val, 'best_test': best_test, 'best_epoch': best_epoch, 'output_dir': str(out_dir),
    }
    write_json(out_dir / 'results.json', result)
    plot_training_curves(metrics_path, out_dir, title='LightGCN-GRAND')
    run_logger.info('Final: %s', result)
    print(f"lightgcn_grand: best_val={best_val:.4f}  best_test={best_test:.4f}")
    return result

In [ ]:
args_lightgcn = SimpleNamespace(
    K=4, hidden=64, dropnode=0.5, dropout=0.5, S=2, lam=1.0, temp=0.5, conf=0.7, warmup=60.0,
    prop_space='logits', learn_gamma=True, use_H=True, Kc=2,
    lr=0.01, l2_reg=5e-4, epochs=epochs_for(250), patience=50, min_epochs=120,
    low_label_split=False, low_label_train_per_class=20, low_label_valid_per_class=30,
    seed=0,
)
res_lightgcn = run_lightgcn_grand(args_lightgcn, run_id=0, device=DEVICE)
results.append(res_lightgcn)

## Summary

In [ ]:
summary_df = pd.DataFrame(results)[['model_family', 'num_params', 'best_val', 'best_test', 'best_epoch', 'output_dir']]
summary_df = summary_df.sort_values('best_test', ascending=False).reset_index(drop=True)
display(summary_df[['model_family', 'num_params', 'best_val', 'best_test', 'best_epoch']])

fig, axes = plt.subplots(1, 2, figsize=(max(11, 1.6 * len(summary_df)), 5))
axes[0].bar(summary_df['model_family'], summary_df['best_test'], color='#3B6D11')
axes[0].set_ylabel('Test accuracy')
axes[0].set_title('Test accuracy')
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis='x', rotation=30)
for tick in axes[0].get_xticklabels():
    tick.set_ha('right')

axes[1].bar(summary_df['model_family'], summary_df['num_params'], color='#854F0B')
axes[1].set_ylabel('Trainable parameters')
axes[1].set_title('Model size')
axes[1].set_yscale('log')
axes[1].tick_params(axis='x', rotation=30)
for tick in axes[1].get_xticklabels():
    tick.set_ha('right')

fig.suptitle(f"Amazon Computers{' (SMOKE_TEST)' if SMOKE_TEST else ''}")
plt.tight_layout()
fig.savefig(OUTPUTS_ROOT / 'comparison_chart.png', dpi=150)
plt.show()

### Accuracy vs. model size

Is a bigger model actually buying more accuracy, or just costing more to train?

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for _, row in summary_df.iterrows():
    ax.scatter(row['num_params'], row['best_test'], s=80)
    ax.annotate(row['model_family'], (row['num_params'], row['best_test']), textcoords='offset points', xytext=(6, 4), fontsize=8)
ax.set_xscale('log')
ax.set_xlabel('Trainable parameters (log scale)')
ax.set_ylabel('Test accuracy')
ax.set_title('Accuracy vs. model size')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Loss and accuracy curves — all models overlaid

Reads each run's `metrics.jsonl` (already written by every `run_x(...)` call above) and plots train loss and validation accuracy on a shared x-axis (epoch), so convergence speed and stability are directly comparable.

In [ ]:
def load_metrics(output_dir):
    path = Path(output_dir) / 'metrics.jsonl'
    if not path.exists():
        return pd.DataFrame()
    records = [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]
    return pd.DataFrame(records)


fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(13, 5))
for res in results:
    metrics_df = load_metrics(res['output_dir'])
    if metrics_df.empty:
        continue
    label = res['model_family']
    ax_loss.plot(metrics_df['epoch'], metrics_df['loss'], label=label, linewidth=1.4)
    if 'val_acc' in metrics_df:
        val_df = metrics_df.dropna(subset=['val_acc'])
        ax_acc.plot(val_df['epoch'], val_df['val_acc'], label=label, linewidth=1.4)

ax_loss.set_xlabel('Epoch'); ax_loss.set_ylabel('Train loss'); ax_loss.set_title('Loss')
ax_loss.legend(fontsize=8)
ax_loss.grid(alpha=0.3)

ax_acc.set_xlabel('Epoch'); ax_acc.set_ylabel('Validation accuracy'); ax_acc.set_title('Validation accuracy')
ax_acc.legend(fontsize=8)
ax_acc.grid(alpha=0.3)

plt.tight_layout()
fig.savefig(OUTPUTS_ROOT / 'comparison_curves.png', dpi=150)
plt.show()